In [1]:
!pip install datasets


In [2]:
!pip install requests

In [4]:
from datasets import load_dataset
import requests
from PIL import Image
import pandas as pd
from pathlib import Path
 
# Load dataset from HuggingFace
print("Loading product dataset...")
try:
    # Try loading the dataset
    dataset = load_dataset("ashraq/fashion-product-images-small", split="train[:100]")  # First 100 samples
    print(f"✓ Loaded {len(dataset)} products")
    
    # Convert to pandas for easier manipulation
    products_df = pd.DataFrame(dataset)
    print(f"Dataset columns: {products_df.columns.tolist()}")
    
except Exception as e:
    print(f"⚠ Could not load HuggingFace dataset: {e}")
    print("Using local images instead...")
    
    # Alternative: Use local images
    # Create a products.json file with product information
    products_data = [
        {
            "id": 1,
            "name": "Wireless Headphones",
            "price": 79.99,
            "category": "Electronics",
            "image_path": "images/product1.jpg"
        },
        # Add more products...
    ]
    
    products_df = pd.DataFrame(products_data)
 
# Create images directory
images_dir = Path("product_images")
images_dir.mkdir(exist_ok=True)
 
print(f"\n✓ Dataset prepared!")
print(f"  Total products: {len(products_df)}")

c:\Users\bbars\miniconda3_new\envs\bootcamp-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading product dataset...


✓ Loaded 100 products
Dataset columns: ['id', 'gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'season', 'year', 'usage', 'productDisplayName', 'image']

✓ Dataset prepared!
  Total products: 100


In [5]:
print(products_df.columns.tolist())


['id', 'gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'season', 'year', 'usage', 'productDisplayName', 'image']


In [6]:
import base64
from io import BytesIO

def encode_image_to_base64(image):
    """Encode a PIL image to a base64 string."""
    buffer = BytesIO()
    image.save(buffer, format="JPEG")
    encoded = base64.b64encode(buffer.getvalue()).decode("utf-8")
    return encoded

# Example usage
sample_image = products_df.iloc[0]["image"]
encoded_image = encode_image_to_base64(sample_image)

print(f"Encoded image length: {len(encoded_image)} characters")
print(f"Encoded prefix: {encoded_image[:40]}...")

Encoded image length: 2388 characters
Encoded prefix: /9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcG...


In [7]:
print("Image ready:", len(encoded_image) > 0)

Image ready: True


Creating Product Listing Prompt

In [8]:
def create_product_listing_prompt(product_name, price, category, additional_info=None):
    """
    Create a prompt for generating product listings.
    
    Parameters:
    - product_name: Name of the product
    - price: Price of the product
    - category: Product category
    - additional_info: Optional additional information
    
    Returns:
    - Formatted prompt string
    """
    prompt = f"""You are an expert e-commerce copywriter. Analyze the product image and create a compelling product listing.
 
Product Information:
- Name: {product_name}
- Price: ${price:.2f}
- Category: {category}
{f'- Additional Info: {additional_info}' if additional_info else ''}
 
Please create a professional product listing that includes:
 
1. **Product Title** (catchy, SEO-friendly, 60 characters max)
2. **Product Description** (detailed, 150-200 words)
   - Highlight key features and benefits
   - Use persuasive language
   - Include relevant details visible in the image
3. **Key Features** (bullet points, 5-7 items)
4. **SEO Keywords** (comma-separated, 10-15 relevant keywords)
 
Format your response as JSON with the following structure:
{{
    "title": "Product title here",
    "description": "Full description here",
    "features": ["Feature 1", "Feature 2", ...],
    "keywords": "keyword1, keyword2, ..."
}}
 
Be specific about what you see in the image. Mention colors, materials, design elements, and any distinctive features."""
    
    return prompt
 
# Test prompt creation
test_prompt = create_product_listing_prompt(
    product_name="Wireless Bluetooth Headphones",
    price=79.99,
    category="Electronics",
    additional_info="Noise cancelling, 30-hour battery"
)
 
print("\n" + "="*50)
print("PROMPT TEMPLATE")
print("="*50)
print(test_prompt[:500] + "...")  # Show first 500 characters


PROMPT TEMPLATE
You are an expert e-commerce copywriter. Analyze the product image and create a compelling product listing.

Product Information:
- Name: Wireless Bluetooth Headphones
- Price: $79.99
- Category: Electronics
- Additional Info: Noise cancelling, 30-hour battery

Please create a professional product listing that includes:

1. **Product Title** (catchy, SEO-friendly, 60 characters max)
2. **Product Description** (detailed, 150-200 words)
   - Highlight key features and benefits
   - Use persuasive ...


In [9]:
test_prompt = create_product_listing_prompt(
    product_name=products_df.iloc[0]["productDisplayName"],
    price=79.99,
    category=products_df.iloc[0]["masterCategory"]
)

print("Prompt ready:", len(test_prompt) > 0)

Prompt ready: True


In [18]:
response = client.responses.create(
    model="gpt-4o-mini",
    input=[
        {
            "role": "user",
            "content": [
                {
                    "type": "input_text",
                    "text": test_prompt
                },
                {
                    "type": "input_image",
                    "image_url": f"data:image/jpeg;base64,{encoded_image}"
                }
            ]
        }
    ]
)

print(response.output_text)

```json
{
    "title": "Stylish Navy Blue Turtle Check Shirt for Men",
    "description": "Elevate your wardrobe with the Turtle Check Men’s Navy Blue Shirt, an ideal blend of sophistication and casual charm. Crafted from high-quality cotton fabric, this shirt offers ultimate comfort paired with a contemporary design. The eye-catching navy blue color showcases a classic check pattern that adds a timeless touch, making it perfect for various occasions—whether you're at the office or out with friends. The short sleeves and tailored fit ensure ease of movement, while the modern collar adds a stylish flair. Elevate your everyday look with this versatile shirt that pairs seamlessly with jeans or chinos. Make a statement effortlessly with the Turtle Check Shirt!",
    "features": [
        "High-quality cotton fabric for all-day comfort",
        "Eye-catching navy blue with classic check pattern",
        "Tailored fit for a modern silhouette",
        "Short sleeves for a relaxed yet styli

Calling the ChatGPT API with Vision

In [19]:
%pip install -U openai

Note: you may need to restart the kernel to use updated packages.


In [22]:
import os
from openai import OpenAI

In [13]:
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [21]:
import os

print("API key found:", bool(os.getenv("OPENAI_API_KEY")))

API key found: True


In [20]:
import os

api_key = os.getenv("OPENAI_API_KEY")
print("API key found:", api_key is not None)

API key found: True


In [23]:
from openai import OpenAI

client = OpenAI()

print("OpenAI client created successfully!")

OpenAI client created successfully!


In [24]:
import os

api_key = os.getenv("OPENAI_API_KEY")

print("API key found:", api_key is not None)
print("Key length:", len(api_key) if api_key else 0)

API key found: True
Key length: 167


In [25]:
from openai import OpenAI

client = OpenAI(api_key=api_key)

print("Client recreated")

Client recreated


In [26]:
import os
from openai import OpenAI

api_key = os.getenv("OPENAI_API_KEY")

print("API key found:", api_key is not None)

API key found: True


In [27]:
client = OpenAI(api_key=api_key)

print("Client created successfully!")

Client created successfully!


In [28]:
import base64
from io import BytesIO

sample_image = products_df.iloc[0]["image"]

buffer = BytesIO()
sample_image.save(buffer, format="JPEG")

encoded_image = base64.b64encode(
    buffer.getvalue()
).decode("utf-8")

print("Image ready:", len(encoded_image) > 0)

Image ready: True


In [29]:
def create_product_listing_prompt(product_name, price, category, additional_info=None):
    prompt = f"""You are an expert e-commerce copywriter. Analyze the product image and create a compelling product listing.

Product Information:
- Name: {product_name}
- Price: ${price:.2f}
- Category: {category}
{f'- Additional Info: {additional_info}' if additional_info else ''}

Please create a professional product listing that includes:

1. Product Title (catchy, SEO-friendly, 60 characters max)
2. Product Description (detailed, 150-200 words)
3. Key Features (bullet points, 5-7 items)
4. SEO Keywords (comma-separated, 10-15 relevant keywords)

Format your response as JSON with the following structure:
{{
    "title": "Product title here",
    "description": "Full description here",
    "features": ["Feature 1", "Feature 2"],
    "keywords": "keyword1, keyword2"
}}

Be specific about what you see in the image. Mention colors, materials, design elements, and distinctive features."""

    return prompt

In [31]:
test_prompt = create_product_listing_prompt(
    product_name=products_df.iloc[0]["productDisplayName"],
    price=79.99,
    category=products_df.iloc[0]["masterCategory"]
)

print("Prompt ready:", len(test_prompt) > 0)


Prompt ready: True


In [25]:
response = client.responses.create(
    model="gpt-4o-mini",
    input=[
        {
            "role": "user",
            "content": [
                {
                    "type": "input_text",
                    "text": test_prompt
                },
                {
                    "type": "input_image",
                    "image_url": f"data:image/jpeg;base64,{encoded_image}"
                }
            ]
        }
    ]
)

print(response.output_text)

```json
{
    "title": "Stylish Navy Blue Turtle Check Shirt for Men",
    "description": "Elevate your wardrobe with the Turtle Check Men Navy Blue Shirt. Designed for the modern man, this shirt features a classic turtle check pattern, combining timeless appeal with contemporary style. Made from high-quality, breathable fabric, it ensures comfort whether you're at the office or out with friends. The rich navy blue color offers versatility, easily pairing with jeans or chinos for any occasion. Short sleeves enhance breathability, making it perfect for warm days. The structured collar adds a polished touch, ensuring you look sharp wherever you go. Make a statement with this stylish addition to your collection!",
    "features": [
        "Premium quality breathable fabric for all-day comfort",
        "Classic turtle check pattern for a stylish look",
        "Rich navy blue color complements various outfits",
        "Short sleeves ideal for casual or formal wear",
        "Structured 

In [32]:
import json

sample_response = """
{
    "title": "Stylish Casual Fashion Item",
    "description": "A versatile and stylish product designed for everyday use. Its clean design and attractive color make it easy to pair with different outfits. The product offers a practical combination of comfort, functionality, and modern style.",
    "features": [
        "Modern design",
        "Attractive color",
        "Versatile everyday use",
        "Comfortable construction",
        "Easy to style",
        "Practical design"
    ],
    "keywords": "fashion, casual, stylish, modern, everyday wear, clothing, trendy, comfortable"
}
"""

product_listing = json.loads(sample_response)

print("Title:", product_listing["title"])
print("\nDescription:", product_listing["description"])

print("\nFeatures:")
for feature in product_listing["features"]:
    print("-", feature)

print("\nKeywords:", product_listing["keywords"])

Title: Stylish Casual Fashion Item

Description: A versatile and stylish product designed for everyday use. Its clean design and attractive color make it easy to pair with different outfits. The product offers a practical combination of comfort, functionality, and modern style.

Features:
- Modern design
- Attractive color
- Versatile everyday use
- Comfortable construction
- Easy to style
- Practical design

Keywords: fashion, casual, stylish, modern, everyday wear, clothing, trendy, comfortable


Step 6: Processing Multiple Products


In [33]:
products_to_process = products_df.head(5).copy()

print("Products to process:", len(products_to_process))

Products to process: 5


In [34]:
import json

def generate_product_listing(product):
    try:
        product_name = product["productDisplayName"]
        category = product["masterCategory"]

        # Create the prompt
        prompt = create_product_listing_prompt(
            product_name=product_name,
            price=79.99,
            category=category
        )

        # Simulated API response for testing
        sample_response = {
            "title": product_name,
            "description": f"A stylish {category.lower()} product designed for everyday use. "
                           f"This product features a modern design and versatile style.",
            "features": [
                "Modern design",
                "Versatile everyday use",
                "Attractive appearance",
                "Practical design",
                "Easy to style"
            ],
            "keywords": f"{category}, fashion, stylish, modern, everyday, product"
        }

        return sample_response

    except Exception as e:
        print(f"Error processing {product_name}: {e}")
        return None

In [35]:
results = []

for index, product in products_to_process.iterrows():
    print(f"Processing product {index + 1} of {len(products_to_process)}...")

    listing = generate_product_listing(product)

    if listing is not None:
        results.append({
            "id": product["id"],
            "product_name": product["productDisplayName"],
            "listing": listing
        })

print("\nProcessing complete!")
print("Successful products:", len(results))

Processing product 1 of 5...
Processing product 2 of 5...
Processing product 3 of 5...
Processing product 4 of 5...
Processing product 5 of 5...

Processing complete!
Successful products: 5


In [36]:
with open("product_listings.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("Results saved to product_listings.json")

Results saved to product_listings.json


In [37]:
with open("product_listings.json", "r", encoding="utf-8") as f:
    saved_results = json.load(f)

print("Saved listings:", len(saved_results))
print("\nFirst product:")
print(saved_results[0])

Saved listings: 5

First product:
{'id': 15970, 'product_name': 'Turtle Check Men Navy Blue Shirt', 'listing': {'title': 'Turtle Check Men Navy Blue Shirt', 'description': 'A stylish apparel product designed for everyday use. This product features a modern design and versatile style.', 'features': ['Modern design', 'Versatile everyday use', 'Attractive appearance', 'Practical design', 'Easy to style'], 'keywords': 'Apparel, fashion, stylish, modern, everyday, product'}}
